# CATAI — Sprite Generation Notebook

Génère tous les sprites pixel art pour CATAI via Stable Diffusion XL.

**Pré-requis :** Runtime > Change runtime type > **T4 GPU**

**Workflow :**
1. Cellule 1 : installation (5 min)
2. Cellule 2 : chargement des modèles (3 min)
3. Cellule 3 : génération des sprites de référence (idle) → **tu valides visuellement**
4. Cellule 4 : génération batch de toutes les frames (~1h30)
5. Cellule 5 : zip et téléchargement

Résultats nommés `<anim>__<dir>__<frame>.png` → à passer dans `scripts/sd_postprocess.py` en local.

In [ ]:
# ── Cellule 1 : Installation ──────────────────────────────────────────────────
!pip install -q diffusers transformers accelerate safetensors xformers controlnet-aux
!pip install -q huggingface_hub Pillow
print('✓ Installation terminée')

In [ ]:
# ── Cellule 2 : Chargement des modèles ───────────────────────────────────────
import torch, os
from diffusers import (
    StableDiffusionXLControlNetPipeline,
    ControlNetModel,
    AutoencoderKL,
)
from diffusers.utils import load_image
from PIL import Image
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16
print(f'Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0) if DEVICE=="cuda" else "none"}')

# ControlNet Canny pour SDXL
print('Chargement ControlNet...')
controlnet = ControlNetModel.from_pretrained(
    'diffusers/controlnet-canny-sdxl-1.0',
    torch_dtype=DTYPE,
)

# VAE amélioré
vae = AutoencoderKL.from_pretrained(
    'madebyollin/sdxl-vae-fp16-fix',
    torch_dtype=DTYPE,
)

# Pipeline SDXL + ControlNet
print('Chargement SDXL...')
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    controlnet=controlnet,
    vae=vae,
    torch_dtype=DTYPE,
    use_safetensors=True,
).to(DEVICE)

# LoRA pixel art (nerijs/pixel-art-xl — libre d'utilisation)
print('Chargement LoRA pixel art...')
pipe.load_lora_weights('nerijs/pixel-art-xl', weight_name='pixel-art-xl.safetensors')

# IP-Adapter SDXL pour cohérence inter-frames
print('Chargement IP-Adapter...')
pipe.load_ip_adapter(
    'h94/IP-Adapter',
    subfolder='sdxl_models',
    weight_name='ip-adapter_sdxl.bin',
)
pipe.set_ip_adapter_scale(0.6)  # 0=ignore référence, 1=copie exacte

pipe.enable_model_cpu_offload()  # économise la VRAM
print('✓ Modèles prêts')

In [ ]:
# ── Cellule 3 : Configuration ─────────────────────────────────────────────────
import cv2
from controlnet_aux import CannyDetector
from IPython.display import display
import random

OUT_DIR  = '/content/catai_sprites_raw'
REF_DIR  = '/content/catai_references'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(REF_DIR, exist_ok=True)

GEN_SIZE = 512    # taille de génération SD
STEPS    = 30
CFG      = 7.5

NEGATIVE = (
    'blurry, antialiasing, gradient, realistic, 3d, photograph, '
    'multiple cats, text, watermark, deformed, ugly, low quality'
)

# Descriptions de chaque personnage
CHARACTERS = {
    'orange_tabby': (
        'cute orange tabby cat, striped fur, white belly, green eyes, '
        'pixel art style, simple design, limited palette'
    ),
    'black_cat': (
        'cute black cat, solid black fur, bright yellow eyes, small white whiskers, '
        'pixel art style, simple design, limited palette'
    ),
    'grey_tuxedo': (
        'cute grey and white tuxedo cat, white chest and paws, blue eyes, '
        'pixel art style, simple design, limited palette'
    ),
    'brown_tabby': (
        'cute brown tabby cat, warm brown stripes, light muzzle, amber eyes, '
        'pixel art style, simple design, limited palette'
    ),
    'calico': (
        'cute calico cat, orange and black patches on white fur, golden eyes, '
        'pixel art style, simple design, limited palette'
    ),
    'siamese': (
        'cute siamese cat, cream body, dark brown face mask and ears, blue eyes, '
        'pixel art style, simple design, limited palette'
    ),
}

# Seeds fixes par personnage (reproductibilité)
SEEDS = {
    'orange_tabby': 42,
    'black_cat':    1337,
    'grey_tuxedo':  2024,
    'brown_tabby':  7777,
    'calico':       31415,
    'siamese':      9999,
}

canny = CannyDetector()

def make_canny(pil_img: Image.Image) -> Image.Image:
    """Extrait les contours Canny d'un sprite PIL pour ControlNet."""
    arr = np.array(pil_img.convert('RGB'))
    edges = cv2.Canny(arr, 50, 150)
    return Image.fromarray(edges).convert('RGB').resize((GEN_SIZE, GEN_SIZE), Image.NEAREST)

def generate(prompt, control_img, ref_img=None, seed=42, cn_scale=0.6):
    """Génère une image avec SDXL + ControlNet (+ IP-Adapter si ref_img fourni)."""
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    kwargs = dict(
        prompt=prompt,
        negative_prompt=NEGATIVE,
        image=control_img,
        controlnet_conditioning_scale=cn_scale,
        num_inference_steps=STEPS,
        guidance_scale=CFG,
        generator=generator,
        width=GEN_SIZE, height=GEN_SIZE,
    )
    if ref_img is not None:
        kwargs['ip_adapter_image'] = ref_img
    result = pipe(**kwargs)
    return result.images[0]

print('✓ Configuration prête')
print(f'Personnages : {list(CHARACTERS.keys())}')

In [ ]:
# ── Cellule 4 : Génération des sprites de référence (idle) ────────────────────
# Génère idle south pour chaque personnage sans ControlNet.
# VALIDE VISUELLEMENT avant de lancer la cellule 5.

# Image blanche vide = pas de contrainte de pose pour l'idle
blank_control = Image.new('RGB', (GEN_SIZE, GEN_SIZE), (255, 255, 255))

for char_id, char_desc in CHARACTERS.items():
    seed = SEEDS[char_id]
    prompt = f'{char_desc}, sitting, facing viewer, white background, centered'
    img = generate(prompt, blank_control, ref_img=None, seed=seed, cn_scale=0.1)
    ref_path = os.path.join(REF_DIR, f'{char_id}__idle.png')
    img.save(ref_path)
    print(f'  {char_id}:')
    display(img.resize((256, 256), Image.NEAREST))

print('\n⚠ Vérifie les sprites ci-dessus.')
print('Si un personnage ne te plaît pas, modifie son SEED ou sa description dans la cellule 3 et relance.')
print('Quand tout est OK → lance la cellule 5.')

In [ ]:
# ── Cellule 5 : Génération batch de toutes les animations ─────────────────────
#
# Chaque animation est définie par :
#  - ses frames PIL existantes (utilisées comme poses ControlNet)
#  - le nombre de frames et les directions
#
# Les sprites PIL sont uploadés dans /content/catai_sprites_pil/
# (upload le dossier catai_linux/cute_orange_cat/animations/ depuis ton poste)

PIL_ANIM_DIR = '/content/catai_sprites_pil'  # à uploader

ANIMATIONS = [
    # (nom, directions, n_frames, desc_pose)
    ('running',            ['east', 'west'], 8,  'running, galloping, side view'),
    ('eating',             ['south'],        11, 'eating from bowl, crouched, front view'),
    ('drinking',           ['south'],        8,  'drinking water, crouched, front view'),
    ('angry',              ['south'],        9,  'angry hissing, arched back, front view'),
    ('waking-getting-up',  ['south'],        9,  'stretching, waking up, yawning'),
    ('sleeping-ball',      ['south'],        4,  'sleeping curled in a ball'),
    ('chasing-mouse',      ['east', 'west'], 6,  'sneaking, nose to ground, side view'),
    ('playing-ball',       ['south'],        8,  'standing, playful pose, front view'),
    ('butterfly',          ['south'],        8,  'reaching up with one paw, standing on hind legs'),
    ('scratching-tree',    ['east', 'west'], 6,  'scratching with front paws, side view'),
    ('peeing',             ['east', 'west'], 6,  'one hind leg raised, side view'),
    ('pooping',            ['south'],        6,  'squatting, back to viewer'),
    # Nouvelles animations
    ('jumping',            ['south'],        6,  'jumping, mid-air, front view'),
    ('climbing',           ['east', 'west'], 8,  'climbing over ledge, hanging'),
    ('love',               ['south'],        6,  'happy, loaf pose, content, front view'),
    ('flat',               ['south'],        4,  'flattened on ground, ears back, front view'),
    ('grooming',           ['south'],        8,  'licking paw, grooming face'),
    ('rolling',            ['south'],        8,  'rolling on back, paws in air'),
    ('surprised',          ['east', 'west'], 5,  'startled, puffed up, side view'),
]

def load_pil_frame(anim, direction, frame_idx):
    """Charge un sprite PIL existant comme référence de pose."""
    path = os.path.join(PIL_ANIM_DIR, anim, direction, f'frame_{frame_idx:03d}.png')
    if os.path.exists(path):
        return Image.open(path).convert('RGBA')
    # Fallback : image blanche (nouvelle animation sans sprite PIL)
    return Image.new('RGBA', (68, 68), (255, 255, 255, 255))

total = sum(len(dirs) * n for _, dirs, n, _ in ANIMATIONS) * len(CHARACTERS)
done  = 0
print(f'Total frames à générer : {total}')
print('─' * 50)

for char_id, char_desc in CHARACTERS.items():
    seed_base = SEEDS[char_id]
    ref_img = Image.open(os.path.join(REF_DIR, f'{char_id}__idle.png'))

    for anim_name, directions, n_frames, pose_desc in ANIMATIONS:
        for direction in directions:
            for fi in range(n_frames):
                out_name = f'{char_id}__{anim_name}__{direction}__{fi:03d}.png'
                out_path = os.path.join(OUT_DIR, out_name)

                if os.path.exists(out_path):
                    done += 1
                    continue  # reprise après interruption

                # Pose ControlNet depuis sprite PIL existant
                pil_frame = load_pil_frame(anim_name, direction, fi)
                control   = make_canny(pil_frame)

                prompt = (
                    f'{char_desc}, {pose_desc}, frame {fi+1} of {n_frames}, '
                    f'white background, centered, pixel art'
                )
                # Seed unique par frame mais reproductible
                seed = seed_base + hash(f'{anim_name}{direction}{fi}') % 10000

                img = generate(prompt, control, ref_img=ref_img, seed=seed)
                img.save(out_path)

                done += 1
                if done % 10 == 0:
                    print(f'  [{done}/{total}] {char_id} / {anim_name} / {direction} / frame {fi}')

print(f'\n✓ Génération terminée : {done} frames dans {OUT_DIR}')

In [ ]:
# ── Cellule 6 : Sauvegarde sur Google Drive ───────────────────────────────────
from google.colab import drive
import shutil

drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/catai_sprites_raw'
shutil.copytree(OUT_DIR,  DRIVE_OUT,  dirs_exist_ok=True)
shutil.copytree(REF_DIR,  DRIVE_OUT + '_references', dirs_exist_ok=True)

print(f'✓ Sprites sauvegardés dans Google Drive : {DRIVE_OUT}')
print()
print('Ensuite sur ton poste :')
print('  pip install rembg pillow')
print('  python3 scripts/sd_postprocess.py <dossier_téléchargé> catai_linux/<perso>')